# Multimodal Pipeline Walkthrough

This notebook runs the multimodal `dre_production` workflow from configured EHR, MRI, and EEG inputs through patient-level fusion, dataset profiling, optional baseline modeling, artifact registration, manifest writing, and reporting summaries.

`likely_dre` is a computable inference from available longitudinal patient data. It is not a formal adjudicated ILAE drug-resistant epilepsy diagnosis.

The notebook is intentionally thin. It configures package modules, calls the end-to-end orchestration runner, and displays concise outputs. Clinical evidence construction, feature extraction, fusion, profiling, modeling, and reporting logic live in package code.

## Architecture Summary

The multimodal workflow is package-driven:

1. The EHR pipeline builds a patient-level table with the computable `likely_dre` inference.
2. The MRI pipeline builds subject-level handcrafted MRI features.
3. The EEG pipeline builds patient-level handcrafted EEG features.
4. The fusion builder aligns available modality outputs by patient identifier and records modality availability.
5. The profiler summarizes the fused patient-level dataset.
6. The optional multimodal baseline model trains a transparent baseline classifier with conservative leakage checks.
7. Manifest, registry, and reporting writers persist run artifacts.

## Environment and Import Setup

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find project root containing pyproject.toml and src/.")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports" / "multimodal_walkthrough"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "multimodal_walkthrough"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Imports

In [ ]:
from datetime import UTC, datetime

import pandas as pd

from core.contracts import RunContextRecord
from datasets import (
    EEGPipelineInputConfig,
    EHRPipelineInputConfig,
    MRIPipelineInputConfig,
    MultimodalPipelineInputConfig,
    MultimodalPipelineRunner,
)
from ingestion import TableLoadRequest

## Run Context Setup

Edit the dataset name and output paths for your local run.

In [ ]:
DATASET_NAME = "multimodal_patient_dataset_walkthrough"

run_context = RunContextRecord(
    run_id=f"multimodal-walkthrough-{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}",
    project_name="dre_production",
    stage_name="multimodal_pipeline_walkthrough",
    seed=42,
    timestamp_utc=datetime.now(UTC),
    metadata={"notebook": "multimodal_pipeline_walkthrough.ipynb"},
)

ehr_output_dir = REPORTS_DIR / "ehr_pipeline"
mri_output_dir = REPORTS_DIR / "mri_pipeline"
eeg_output_dir = REPORTS_DIR / "eeg_pipeline"
multimodal_output_dir = REPORTS_DIR / "multimodal_pipeline"

run_context

## Input Configuration

Replace these example paths with local extracts. If a modality is intentionally unavailable, set that modality config to `None`. If a config is provided, the referenced files or folders must exist and the pipeline will fail clearly when they do not.

In [ ]:
diagnoses_path = DATA_DIR / "ehr" / "diagnoses.csv"
medications_path = DATA_DIR / "ehr" / "medications.csv"
seizure_events_path = DATA_DIR / "ehr" / "seizure_events.csv"
mri_dataset_root = DATA_DIR / "mri"
eeg_dataset_root = DATA_DIR / "eeg"

ehr_input_config = EHRPipelineInputConfig(
    dataset_name=f"{DATASET_NAME}_ehr",
    diagnoses_request=TableLoadRequest(
        path=str(diagnoses_path),
        required_columns=["patient_id", "diagnosis_code"],
        parse_date_columns=["diagnosis_time"],
    ),
    medications_request=TableLoadRequest(
        path=str(medications_path),
        required_columns=["patient_id", "medication_name"],
        parse_date_columns=["medication_start", "medication_end"],
    ),
    seizure_events_request=TableLoadRequest(
        path=str(seizure_events_path),
        required_columns=["patient_id"],
        parse_date_columns=["event_time"],
    ),
    output_dir=str(ehr_output_dir),
)

mri_input_config = MRIPipelineInputConfig(
    dataset_name=f"{DATASET_NAME}_mri",
    dataset_root=str(mri_dataset_root),
    output_dir=str(mri_output_dir),
)

eeg_input_config = EEGPipelineInputConfig(
    dataset_name=f"{DATASET_NAME}_eeg",
    dataset_root=str(eeg_dataset_root),
    output_dir=str(eeg_output_dir),
)

multimodal_input_config = MultimodalPipelineInputConfig(
    dataset_name=DATASET_NAME,
    output_dir=str(multimodal_output_dir),
    ehr_input_config=ehr_input_config,
    mri_input_config=mri_input_config,
    eeg_input_config=eeg_input_config,
    run_multimodal_model=True,
    notes="Generated by thin multimodal walkthrough notebook.",
)

multimodal_input_config.to_dict()

## Required Input Check

This cell fails before execution if configured example inputs are missing. Set a modality config to `None` when that modality should be intentionally omitted.

In [ ]:
required_paths = {}
if multimodal_input_config.ehr_input_config is not None:
    required_paths.update(
        {
            "ehr_diagnoses": diagnoses_path,
            "ehr_medications": medications_path,
            "ehr_seizure_events": seizure_events_path,
        }
    )
if multimodal_input_config.mri_input_config is not None:
    required_paths["mri_dataset_root"] = mri_dataset_root
if multimodal_input_config.eeg_input_config is not None:
    required_paths["eeg_dataset_root"] = eeg_dataset_root

missing_paths = {name: str(path) for name, path in required_paths.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError(
        "Update the multimodal input paths or set unavailable modality configs to None. Missing paths: "
        + json.dumps(missing_paths, indent=2)
    )

required_paths

## Multimodal Pipeline Execution

This cell delegates modality runs, fusion, profiling, optional baseline modeling, artifact registration, manifest writing, and reporting bundle generation to package code.

In [ ]:
runner = MultimodalPipelineRunner()
run_result = runner.run(
    input_config=multimodal_input_config,
    run_context=run_context,
)

run_result.to_dict()

## Result Loading and Summary Extraction

The reporting bundle is written by package reporting code. This cell only loads it for display.

In [ ]:
reporting_bundle_path = Path(run_result.reporting_bundle_path)
reporting_bundle = json.loads(reporting_bundle_path.read_text(encoding="utf-8"))

fused_dataset_path = Path(run_result.fused_dataset_path)
fused_df = pd.read_csv(fused_dataset_path)

pipeline_summary = reporting_bundle.get("pipeline_summary") or {}
model_summary = reporting_bundle.get("model_summary") or {}
dataset_profile_summary = reporting_bundle.get("dataset_profile_summary") or {}

{
    "reporting_bundle_path": str(reporting_bundle_path),
    "fused_dataset_path": str(fused_dataset_path),
    "fused_shape": fused_df.shape,
}

## Results Display

In [ ]:
test_metrics = (model_summary.get("key_metrics") or {}) if model_summary else {}

results_display = {
    "fused_dataset_shape": fused_df.shape,
    "fused_row_count": run_result.fused_row_count,
    "patients_with_ehr": run_result.patients_with_ehr,
    "patients_with_mri": run_result.patients_with_mri,
    "patients_with_eeg": run_result.patients_with_eeg,
    "patients_with_all_modalities": run_result.patients_with_all_modalities,
    "multimodal_model_ran": run_result.multimodal_model_path is not None,
    "test_accuracy": test_metrics.get("test_accuracy"),
    "test_f1": test_metrics.get("test_f1"),
    "test_roc_auc": test_metrics.get("test_roc_auc"),
}
results_display

In [ ]:
artifact_paths = {
    "fused_dataset": run_result.fused_dataset_path,
    "fused_profile": run_result.fused_profile_path,
    "artifact_registry": run_result.artifact_registry_path,
    "run_manifest": run_result.run_manifest_path,
    "reporting_bundle": run_result.reporting_bundle_path,
    "multimodal_model": run_result.multimodal_model_path,
    "multimodal_metrics": run_result.multimodal_metrics_path,
}
{key: value for key, value in artifact_paths.items() if value is not None}

In [ ]:
summary_display = {
    "top_missing_columns": dataset_profile_summary.get("top_missing_columns", []),
    "model_features_used": model_summary.get("feature_cols_used", []),
    "leakage_excluded_columns": model_summary.get("leakage_excluded_cols", []),
    "run_notes": run_result.notes,
}
summary_display

## Interpretation and Limitations

`likely_dre` is a computable inference from available records, not formal adjudicated ILAE DRE.

Modality coverage may be uneven. A patient can have EHR evidence without MRI or EEG features, and the fused table records this through modality availability flags.

Fusion quality depends on patient identity alignment across EHR, MRI, and EEG sources. Review linkage assumptions before interpreting multimodal outputs.

The current modeling path is an auditable baseline for development and comparison. It is not a clinically validated deployment system.

## Optional Next Steps

- Strengthen patient identity linkage across source systems.
- Expand MRI and EEG feature representations with validated feature sets.
- Add calibration, threshold analysis, and subgroup performance review.
- Extend dataset insight reports for modality missingness and cohort drift.
- Harden deployment workflows, monitoring, and clinical review processes.